# Selected-Page MBG + Example Agent Test
This notebook runs MBG extraction and the optional Example Agent on a selected slice of `artifacts/deepseek_ocr_fullbook`.

In [6]:
from pathlib import Path
import json
import sys

ROOT = Path('/home/snt/projects/GramMetaRL')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from grammetarl import ExampleAgent, load_cards

OCR_DIR = ROOT / 'artifacts/deepseek_ocr_fullbook'
OCR_PAGES_JSONL = OCR_DIR / 'pages_ocr.jsonl'
GRAMMAR_DIR = ROOT / 'artifacts/mbg_fullbook_qwen_test'
TEST_DIR = ROOT / 'artifacts/mbg_fullbook_qwen_test_examples'
TEST_DIR.mkdir(parents=True, exist_ok=True)

PAGE_START = 36
PAGE_END = 37
LLM_ENDPOINT = 'http://10.6.32.16:8000/v1'
LLM_MODEL = 'nvidia/Qwen3.6-35B-A3B-NVFP4'
MAX_RULES = 10

MBG_SOURCE_JSONL = GRAMMAR_DIR / 'lb_mbg_cards_fullbook.jsonl'
if not MBG_SOURCE_JSONL.exists():
    candidates = sorted(
        p for p in GRAMMAR_DIR.glob('lb_mbg_cards*.jsonl')
        if '_examples' not in p.name and '_with_examples' not in p.name
    )
    if not candidates:
        raise FileNotFoundError(f'No grammar bullet jsonl found in {GRAMMAR_DIR}')
    MBG_SOURCE_JSONL = candidates[0]

EXAMPLE_OUT = TEST_DIR / f'lb_examples_from_grammar_p{PAGE_START}_p{PAGE_END}.jsonl'

print('OCR dir:', OCR_DIR)
print('OCR pages JSONL:', OCR_PAGES_JSONL)
print('Grammar source dir:', GRAMMAR_DIR)
print('Grammar source jsonl:', MBG_SOURCE_JSONL)
print('Selected pages:', PAGE_START, '-', PAGE_END)
print('Example output:', EXAMPLE_OUT)

all_cards = load_cards(MBG_SOURCE_JSONL)
selected_cards = [
    c for c in all_cards
    if c.source.page_start <= PAGE_END and c.source.page_end >= PAGE_START
]
if not selected_cards:
    print('[WARN] No grammar cards matched selected pages; fallback to first cards in source file.')
    selected_cards = all_cards[:MAX_RULES]
else:
    selected_cards = selected_cards[:MAX_RULES]

print('Loaded grammar cards:', len(all_cards))
print('Selected cards for example generation:', len(selected_cards))

page_rows = {}
with OCR_PAGES_JSONL.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if row.get('error'):
            continue
        page = row.get('page_number')
        if isinstance(page, int):
            page_rows[page] = row

agent = ExampleAgent(
    endpoint=LLM_ENDPOINT,
    model=LLM_MODEL,
    timeout=900,
    min_confidence=7,
)

example_records = []
for i, card in enumerate(selected_cards, start=1):
    source_lines = []
    for page in range(card.source.page_start, card.source.page_end + 1):
        row = page_rows.get(page)
        if not row:
            continue
        for block in row.get('blocks', []):
            if not isinstance(block, dict):
                continue
            text = str(block.get('text', '')).strip()
            if not text:
                continue
            label = str(block.get('label') or block.get('block_type') or 'unknown')
            source_lines.append(f'[OCR_BLOCK type={label}] {text}')

    source_text = '\n'.join(source_lines).strip() or (card.source.excerpt or '')
    print(f'[{i}/{len(selected_cards)}] generating example for rule:', card.id)
    try:
        record = agent.generate_record(card, source_text=source_text)
    except Exception as exc:  # noqa: BLE001
        print('  [WARN] example generation failed:', exc)
        continue
    if record is None:
        print('  [INFO] low quality or null output, skipped')
        continue
    example_records.append(record.model_dump(mode='json'))

EXAMPLE_OUT.parent.mkdir(parents=True, exist_ok=True)
with EXAMPLE_OUT.open('w', encoding='utf-8') as f:
    for row in example_records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('\nDone.')
print('Saved example records:', len(example_records))
print('Output file:', EXAMPLE_OUT)

for i, record in enumerate(example_records[:5], start=1):
    print(f"\n[{i}] rule_id={record.get('rule_id')}")
    example = record.get('example', {})
    print('english_source:', example.get('english_source'))
    print('wrong_expression:', example.get('wrong_expression'))
    print('correct_expression:', example.get('correct_expression'))
    print('confidence:', example.get('confidence'))

OCR dir: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook
OCR pages JSONL: /home/snt/projects/GramMetaRL/artifacts/deepseek_ocr_fullbook/pages_ocr.jsonl
Grammar source dir: /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test
Grammar source jsonl: /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test/lb_mbg_cards_fullbook.jsonl
Selected pages: 36 - 37
Example output: /home/snt/projects/GramMetaRL/artifacts/mbg_fullbook_qwen_test_examples/lb_examples_from_grammar_p36_p37.jsonl
[WARN] No grammar cards matched selected pages; fallback to first cards in source file.
Loaded grammar cards: 57
Selected cards for example generation: 10
[1/10] generating example for rule: LB_0001
  [WARN] example generation failed: HTTPConnectionPool(host='10.6.32.16', port=8000): Read timed out. (read timeout=900)
[2/10] generating example for rule: LB_0002
  [INFO] low quality or null output, skipped
[3/10] generating example for rule: LB_0003
  [INFO] low quality or null outpu

KeyboardInterrupt: 